In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import time
import matplotlib.pyplot as plt

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU is available and will be used: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("GPU not found. The model will run on the CPU.")

GPU is available and will be used: NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
DATA_FOLDER = '../Weather_Merged_CSVs'
PLOT_FOLDER = 'GRU_Results'
os.makedirs(PLOT_FOLDER, exist_ok=True)

In [4]:
n_steps_to_test = [7, 14, 21, 30, 60]
all_crops_summary = []

In [5]:
def create_sequences(data, n_steps, target_column):
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data.iloc[i:(i + n_steps)].values)
        y.append(data.iloc[i + n_steps][target_column])
    return np.array(X), np.array(y).reshape(-1, 1)

In [6]:
def calculate_mape(actual, predicted):
    actual, predicted = np.array(actual), np.array(predicted)
    nonzero_mask = actual != 0
    if not np.any(nonzero_mask):
        return float('inf')
    mape = np.mean(np.abs((actual[nonzero_mask] - predicted[nonzero_mask]) / actual[nonzero_mask])) * 100
    return mape

In [7]:
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout_prob):
        super(GRUModel, self).__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout_prob if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout_prob)
        self.fc = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        out, _ = self.gru(x)
        out = self.dropout(out[:, -1, :])
        out = self.fc(out)
        return out

In [ ]:
csv_files = [f for f in os.listdir(DATA_FOLDER) if f.endswith('.csv')]

for csv_file in csv_files:
    try:
        crop_name = os.path.splitext(csv_file)[0]
        file_path = os.path.join(DATA_FOLDER, csv_file)
        
        print("\n" + "="*70)
        print(f"Processing Crop: {crop_name}")
        print("="*70)

        df = pd.read_csv(file_path)
        df['Price Date'] = pd.to_datetime(df['Price Date'])
        df.set_index('Price Date', inplace=True)
        df.sort_index(inplace=True)

        if len(df) < 100:
            print(f"Skipping {crop_name} due to insufficient data ({len(df)} rows).")
            continue

        # Feature Engineering, Encoding, Scaling
        df['day_of_year'] = df.index.dayofyear
        df['week_of_year'] = df.index.isocalendar().week.astype(int)
        df['month'] = df.index.month
        
        categorical_cols = ['District Name', 'Market Name', 'Commodity', 'Variety', 'Grade']
        for col in categorical_cols:
            if df[col].dtype == 'object':
                df[col] = LabelEncoder().fit_transform(df[col])

        scaler = MinMaxScaler(feature_range=(0, 1))
        df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
        target_column = 'Modal Price (Rs./Quintal)'

        # --- Hyperparameter Tuning Loop for the current crop ---
        best_mape_for_crop = float('inf') # MODIFIED: We will now track the best MAPE
        best_n_steps_for_crop = -1
        best_predictions_for_crop = None
        best_actuals_for_crop = None
        best_model_for_crop_state = None 

        for n_steps in n_steps_to_test:
            print(f"\n--- Testing {crop_name} with n_steps = {n_steps} ---")
            
            if len(df_scaled) <= n_steps:
                print(f"Skipping n_steps={n_steps} as it's too large.")
                continue
                
            X, y = create_sequences(df_scaled, n_steps, target_column)
            split = int(0.8 * len(X))
            X_train, X_test, y_train, y_test = X[:split], X[split:], y[:split], y[split:]
            X_train_tensor = torch.from_numpy(X_train).float()
            y_train_tensor = torch.from_numpy(y_train).float()
            X_test_tensor = torch.from_numpy(X_test).float()
            y_test_tensor = torch.from_numpy(y_test).float()
            train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
            test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
            train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
            test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            input_size = X_train.shape[2]
            model = GRUModel(input_size, hidden_size=50, num_layers=2, output_size=1, dropout_prob=0.2).to(device)
            criterion = nn.MSELoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
            num_epochs = 50
            for epoch in range(num_epochs):
                model.train()
                for batch_X, batch_y in train_loader:
                    batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                    outputs = model(batch_X)
                    loss = criterion(outputs, batch_y)
                    optimizer.zero_grad(); loss.backward(); optimizer.step()
            
            model.eval()
            all_predictions = []
            with torch.no_grad():
                for batch_X, _ in test_loader:
                    outputs = model(batch_X.to(device))
                    all_predictions.append(outputs.cpu().numpy())
            predictions = np.concatenate(all_predictions)

            price_col_index = df_scaled.columns.get_loc(target_column)
            dummy_pred = np.zeros((len(predictions), df_scaled.shape[1])); dummy_pred[:, price_col_index] = predictions.flatten()
            inversed_predictions = scaler.inverse_transform(dummy_pred)[:, price_col_index]
            dummy_actual = np.zeros((len(y_test), df_scaled.shape[1])); dummy_actual[:, price_col_index] = y_test.flatten()
            inversed_actual = scaler.inverse_transform(dummy_actual)[:, price_col_index]
            
            mape = calculate_mape(inversed_actual, inversed_predictions)
            
            print(f"n_steps = {n_steps} | MAPE = {mape:.2f}%")

            # MODIFIED: Determine the best model based on the lowest MAPE
            if mape < best_mape_for_crop:
                best_mape_for_crop = mape
                best_n_steps_for_crop = n_steps
                best_predictions_for_crop = inversed_predictions
                best_actuals_for_crop = inversed_actual
                best_model_for_crop_state = model.state_dict()

        # --- After tuning, save the best results for the crop ---
        if best_n_steps_for_crop != -1:
            best_rmse = np.sqrt(mean_squared_error(best_actuals_for_crop, best_predictions_for_crop))
            # MODIFIED: Changed the summary dictionary to reflect the new metrics
            summary = {
                'Crop': crop_name,
                'Best n_steps': best_n_steps_for_crop,
                'Best RMSE': round(best_rmse, 2),
                'Loss % (MAPE)': round(best_mape_for_crop, 2),
                'Accuracy %': round(100 - best_mape_for_crop, 2)
            }
            all_crops_summary.append(summary)

            model_save_path = os.path.join(PLOT_FOLDER, f"{crop_name}_best_model_gru.pth")
            torch.save(best_model_for_crop_state, model_save_path)
            print(f"\nSaved best model for {crop_name} to {model_save_path}")
            
            plt.figure(figsize=(14, 7))
            plt.plot(best_actuals_for_crop, color='red', label='Actual Price')
            plt.plot(best_predictions_for_crop, color='blue', label=f'Predicted Price (Best n_steps={best_n_steps_for_crop})')
            plt.title(f'Best Model Prediction for {crop_name} (MAPE: {best_mape_for_crop:.2f}%)')
            plt.xlabel('Time (Test Set)'); plt.ylabel('Price (Rs./Quintal)'); plt.legend()
            plot_filename = os.path.join(PLOT_FOLDER, f"{crop_name}_prediction_plot.png")
            plt.savefig(plot_filename)
            plt.close()
            print(f"\nSaved best plot for {crop_name} to {plot_filename}")

    except Exception as e:
        print(f"\nCould not process {csv_file}. Error: {e}")
        continue


Processing Crop: Maize-2019-2022

--- Testing Maize-2019-2022 with n_steps = 7 ---
n_steps = 7 | MAPE = 8.18%

--- Testing Maize-2019-2022 with n_steps = 14 ---
n_steps = 14 | MAPE = 7.67%

--- Testing Maize-2019-2022 with n_steps = 21 ---
n_steps = 21 | MAPE = 7.23%

--- Testing Maize-2019-2022 with n_steps = 30 ---
n_steps = 30 | MAPE = 7.26%

--- Testing Maize-2019-2022 with n_steps = 60 ---
n_steps = 60 | MAPE = 7.52%

Saved best model for Maize-2019-2022 to GRU_Results\Maize-2019-2022_best_model_gru_optimized.pth

Saved best plot for Maize-2019-2022 to GRU_Results\Maize-2019-2022_prediction_plot.png


In [ ]:
summary_df = pd.DataFrame(all_crops_summary)
summary_df.sort_values(by='Accuracy %', ascending=False, inplace=True)
summary_filename = './GRU_Results/GRU_Accuracy_Summary.csv'
summary_df.to_csv(summary_filename, index=False)

print("\n\n" + "="*70)
print(f"Processing complete. Summary of results saved to '{summary_filename}'")
print("="*70)
print(summary_df)



Processing complete. Summary of results saved to './GRU_Results/GRU_Accuracy_Summary.csv'
                   Crop  Best n_steps  Best RMSE  Loss % (MAPE)  Accuracy %
13               Garlic            21     710.54           2.65       97.35
5            Cashewnuts            14    2845.57           6.87       93.13
35               Rubber            60    1058.73           6.91       93.09
21      Maize-2019-2022            60     204.19           7.21       92.79
33         Red_Chillies            21    2306.33           8.72       91.28
30       Ragi-2015-2019            30     239.30           9.41       90.59
32       Ragi-2022-2025            30     479.60           9.62       90.38
41             Turmeric            60    1359.56           9.92       90.08
1       Bajra-2022-2025            30     294.59          10.75       89.25
26                Onion            30     443.29          10.93       89.07
12               Cotton            60    1077.60          11.13       88